# Grad-CAM on CPU
Use actual layer4 activations and gradients for the predicted class. Images are shown in the same letterboxed coordinate system used by the model. Influence maps are not lesion localization or a verified diagnostic explanation. Zero maps are shown honestly.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/cpu.yaml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory")
os.chdir(ROOT)
import yaml
from ml.runtime import configure
config = yaml.safe_load(Path("configs/cpu.yaml").read_text())
configure(config["seed"], config["threads"])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ml.inference.predictor import Predictor
from ml.data.transforms import Letterbox
checkpoint = Path("artifacts/checkpoints/run/best.pt")
if checkpoint.exists():
    predictor = Predictor(checkpoint)
    manifest = pd.read_csv(config["manifest"])
    row = manifest[manifest.split == "val"].iloc[0]
    original = Image.open(Path(config["dataset_root"]) / row.path).convert("RGB")
    result, heatmap, combined = predictor.predict(original, explain=True)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, image, title in zip(axes, [Letterbox(predictor.size)(original), heatmap, combined],
                              ["Model input", "Grad-CAM", "Overlay"]):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    plt.show()
    display(result)
else:
    print("No approved trained model. Smoke checkpoints are intentionally excluded.")